# 1. Load reference data and prices from Refinitiv (S&P 500 universe)

This notebook fills the database with everything the backtests need, using **Refinitiv (LSEG)** as the vendor:

1. **Index membership** for `.SPX` over the backtest window (joiners and leavers), mapped onto our `security_master`
   identifiers, plus **float-adjusted shares outstanding** for market-cap weighting.
2. Four **portfolios** with daily holdings:
   `SP500` (the reconstructed index, one row per constituent per day), `SPY` (the tradable ETF), `GSPC` (the
   published index level) and `MAG8` (a concentrated "Magnificent 8" thesis book).
3. **End-of-day prices** for every security those portfolios touch, pulled in small batches with retries, a
   gap "mop-up" pass and a data-quality guard.

All pipeline logic lives in `data_engineering.loaders`; this notebook only sets parameters and calls it.
Notebook **02** consumes the result.

### Prerequisites
* `pip install -e .[refinitiv,dev]` from the project root (no `sys.path` hacks are needed anymore).
* LSEG Workspace running locally (the session is opened lazily on first use).
* SQL Server credentials in the OS keyring under `ihub_sql_connection` (`db`, `uid`, `pwd`).

### Runtime
The constituent + float-share pull and the EOD pull each take 15–25 minutes for the full window.
Both are skipped on re-runs when their target tables already hold data unless `FORCE_RELOAD = True`.

## 1.1 Parameters

In [1]:
START_DATE = "2020-01-01"
END_DATE = "2026-08-31"

INDEX_RIC = ".SPX"          # S&P 500 on Refinitiv
INDEX_ID = 2                # id under which membership is stored in reference.index_constituents
ETF_RIC, INDEX_LEVEL_RIC = "SPY.P", ".SPX"

MAG8 = {"MSFT": 5, "GOOGL": 4, "AVGO": 3, "META": 2, "NVDA": 1, "TSLA": 1, "AMZN": 1, "AAPL": 1}   # ticker -> shares held

RESET_TABLES = True        # True: TRUNCATE every staging table first (clean rebuild)
FORCE_RELOAD = True        # True: re-pull Refinitiv data even when the tables are already populated

## 1.2 Connections

In [2]:
import sys
import os

import pandas as pd

# Get the current notebook's directory and go up to parent
current_dir = os.getcwd()
parent_dir = os.path.dirname(os.path.dirname(current_dir))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print(f"Current directory: {current_dir}")
print(f"Added to path: {parent_dir}")
data_eng_path = os.path.join(parent_dir, 'data_engineering')
files_path = os.path.join(parent_dir, 'toolkit\\files')
print(f"Data engineering folder exists: {os.path.exists(data_eng_path)}")
print(f"Files folder exists: {os.path.exists(files_path)}")

Current directory: c:\SourceCode\InvestmentManagement\toolkit\notebooks
Added to path: c:\SourceCode\InvestmentManagement
Data engineering folder exists: True
Files folder exists: True


In [3]:
import logging

import pandas as pd

from data_engineering.database import database
from data_engineering.index_constituents.refinitiv import load_index_constituents_and_shares
from data_engineering.loaders import (
    build_backtest_universe, coverage_report, ensure_security, get_or_create_portfolio, load_eod_prices,
    mop_up_missing_prices, prune_low_coverage_holdings, resolve_security_ids_by_ric, write_constant_holdings,
    write_index_holdings,
)
from data_engineering.refinitiv import ensure_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
ensure_session()   # LSEG Workspace must be running
engine, connection, conn_str, session = database.get_db_connection()

2026-09-25 13:34:23,855 INFO httpx: HTTP Request: GET http://localhost:9005/api/status "HTTP/1.1 200 OK"
2026-09-25 13:34:23,988 INFO httpx: HTTP Request: POST http://localhost:9005/api/handshake "HTTP/1.1 200 OK"
2026-09-25 13:34:23,991 INFO data_engineering.refinitiv.session: LSEG session opened


Database connection successful.


## 1.3 Optional: reset the staging tables

Only for a clean rebuild. Tables that do not exist in this deployment are skipped.

In [4]:
STAGING_TABLES = [
    "dbo.portfolio", "dbo.portfolio_holdings", "dbo.market_data", "dbo.currency_rates",
    "analytics.portfolio_analytics", "dbo.security_fundamentals", "reference.index_constituents",
    "dbo.security_master", "dbo.security_vendor_xref",
]
if RESET_TABLES:
    print("Truncated:", database.truncate_tables(engine, STAGING_TABLES))
else:
    print("RESET_TABLES is False - existing data kept.")

Truncated: ['dbo.portfolio', 'dbo.portfolio_holdings', 'dbo.market_data', 'dbo.security_fundamentals', 'reference.index_constituents', 'dbo.security_master', 'dbo.security_vendor_xref']


## 1.4 Index membership and float-adjusted shares

`load_index_constituents_and_shares` reconstructs `.SPX` membership from a start-date snapshot plus joiner/leaver
events, resolves each constituent to a `security_id` (exact RIC → normalised RIC → ISIN/SEDOL/CUSIP/FIGI, creating
master rows for unknown names), drops futures tickers the feed sometimes includes (`ES`, `NQ`, ...), pulls
`TR.SharesOutstanding × TR.FreeFloatPct` and writes both tables. Every LSEG call is chunked and retried.

In [5]:
if FORCE_RELOAD or not database.table_has_rows(engine, "index_constituents", schema="reference"):
    constituents, shares = load_index_constituents_and_shares(INDEX_RIC, START_DATE, END_DATE, INDEX_ID, session, engine)
    print(f"{len(constituents)} membership intervals, {len(shares)} float-adjusted share observations written")
else:
    print("index_constituents already populated - skipped (set FORCE_RELOAD = True to reload)")

2026-09-25 13:34:39,873 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:34:43,335 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:34:44,784 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:34:46,874 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:34:54,330 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:34:59,251 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:35:03,736 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:35:03,743 INFO data_engineering.refinitiv.client: get_data progress 4/4 chunks (4 returned data)
2026-09-25 13:35:03,781 INFO data_engineering.index_constituents.refinitiv: creating 637 new security_master rows for unresolved normalized RICs


Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.
Wrote 1 vendor xref rows for Refinitiv.


2026-09-25 13:35:07,546 INFO data_engineering.index_constituents.refinitiv: dropping 1 non-equity ticker(s) from .SPX constituents: ['ES']
2026-09-25 13:35:07,547 INFO data_engineering.index_constituents.refinitiv: RIC->security_id coverage: 637/637 (0 unresolved)
2026-09-25 13:35:07,548 INFO data_engineering.index_constituents.refinitiv: pulling float-adjusted shares for 636 instruments over 2020-01-01..2026-08-31 in chunks of 20
2026-09-25 13:35:10,037 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:35:25,978 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:35:41,586 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:35:57,608 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:36:13,622 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:36:29,676 INFO httpx: HTTP Request: 

Upserting 998660 security_fundamentals rows (non-destructive MERGE)...


2026-09-25 13:47:03,411 INFO data_engineering.index_constituents.refinitiv: wrote 637 constituent rows (index_id=2) and 1067208 shares rows


Security fundamentals data successfully written (998660 rows upserted).
637 membership intervals, 1067208 float-adjusted share observations written


## 1.5 Portfolios and daily holdings

Portfolios are looked up by short name before being created, and each portfolio's holdings are replaced
wholesale, so this section is safe to re-run.

* **SP500** – one row per constituent per business day it was a member; `held_shares = 1`. The market-cap weight is
  computed at backtest time from float-adjusted shares × prior-day price (`analytics.portfolio.weights`).
* **SPY / GSPC** – a single line holding the ETF / the index level, used to validate the reconstruction.
* **MAG8** – a fixed-share thesis book.

In [6]:
spy_id = ensure_security(session, engine, ETF_RIC, "SPDR S&P 500 ETF", "ETF")
gspc_id = ensure_security(session, engine, INDEX_LEVEL_RIC, "S&P 500 Index", "Index")

port_ids = {
    "SP500": get_or_create_portfolio(session, "SP500", "S&P 500 Reconstructed", "Benchmark"),
    "SPY": get_or_create_portfolio(session, "SPY", "S&P 500 ETF", "Portfolio"),
    "GSPC": get_or_create_portfolio(session, "GSPC", "S&P 500 Price Index", "Portfolio"),
    "MAG8": get_or_create_portfolio(session, "MAG8", "Backtest Mag 8", "Portfolio"),
}

n_index = write_index_holdings(session, engine, port_ids["SP500"], INDEX_ID, START_DATE, END_DATE)
n_spy = write_constant_holdings(session, port_ids["SPY"], {spy_id: 1.0}, START_DATE, END_DATE)
n_gspc = write_constant_holdings(session, port_ids["GSPC"], {gspc_id: 1.0}, START_DATE, END_DATE)

mag8_ids = resolve_security_ids_by_ric(session, engine, list(MAG8))
missing = sorted(set(MAG8) - set(mag8_ids))
assert not missing, f"MAG8 tickers not found in security_vendor_xref: {missing}"
n_mag8 = write_constant_holdings(session, port_ids["MAG8"], {mag8_ids[t]: q for t, q in MAG8.items()}, START_DATE, END_DATE)

print(f"holdings rows -> SP500: {n_index:,} | SPY: {n_spy:,} | GSPC: {n_gspc:,} | MAG8: {n_mag8:,}")
database.read_portfolio(session, engine, list(port_ids))

2026-09-25 13:47:03,436 INFO data_engineering.loaders.portfolios: inserted security_master + xref for SPY.P (security_id=638)
2026-09-25 13:47:03,459 INFO data_engineering.loaders.portfolios: inserted security_master + xref for .SPX (security_id=639)
2026-09-25 13:47:03,476 INFO data_engineering.loaders.portfolios: created portfolio SP500 (port_id=1)
2026-09-25 13:47:03,479 INFO data_engineering.loaders.portfolios: created portfolio SPY (port_id=2)
2026-09-25 13:47:03,482 INFO data_engineering.loaders.portfolios: created portfolio GSPC (port_id=3)
2026-09-25 13:47:03,487 INFO data_engineering.loaders.portfolios: created portfolio MAG8 (port_id=4)
2026-09-25 13:47:28,922 INFO data_engineering.loaders.portfolios: wrote 876161 holdings rows for port_id=1
2026-09-25 13:47:29,835 INFO data_engineering.loaders.portfolios: wrote 1739 holdings rows for port_id=2
2026-09-25 13:47:30,486 INFO data_engineering.loaders.portfolios: wrote 1739 holdings rows for port_id=3
2026-09-25 13:47:31,629 INFO

holdings rows -> SP500: 876,161 | SPY: 1,739 | GSPC: 1,739 | MAG8: 13,912


,port_id,portfolio_short_name,portfolio_name,portfolio_type
0,1,SP500,S&P 500 Reconstructed,Benchmark
1,2,SPY,S&P 500 ETF,Portfolio
2,3,GSPC,S&P 500 Price Index,Portfolio
3,4,MAG8,Backtest Mag 8,Portfolio


## 1.6 End-of-day prices

`build_backtest_universe` collects every `.SPX` member (point-in-time), the ETF and index level, and the MAG8
names, strips Refinitiv `^event` suffixes and exchange-qualifies bare tickers (LSEG returns sparse data for
`AAPL`, full history for `AAPL.O`).

`load_eod_prices` then pulls in batches of 20 with up to 5 retries **of only the missing names**, drops rows with
implausible (>25 %) one-day moves, and writes each batch immediately so an interrupted run keeps its progress.

In [7]:
universe = build_backtest_universe(session, engine, INDEX_ID, extra_rics=(ETF_RIC, INDEX_LEVEL_RIC), extra_portfolios=("MAG8",))
print(f"{len(universe)} securities in the EOD universe")

if FORCE_RELOAD or not database.table_has_rows(engine, "market_data"):
    summary = load_eod_prices(universe, session, START_DATE, END_DATE, vendor="refinitiv", batch_size=20, max_retries=5)
    print(summary)
else:
    print("market_data already populated - skipped (set FORCE_RELOAD = True to reload)")

2026-09-25 13:47:32,306 INFO data_engineering.loaders.universe: backtest universe: 638 securities (index_id=2 + 2 extra RICs + ['MAG8'])


638 securities in the EOD universe


2026-09-25 13:47:34,384 INFO data_engineering.loaders.eod_loader: EOD pull: 638 securities in 32 batches of 20
2026-09-25 13:47:47,953 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:47:55,360 INFO data_engineering.eod_data.refinitiv: symbols with no data: ['ABMD.OQ', 'AGN.N']
2026-09-25 13:47:55,370 WARNING data_engineering.loaders.eod_loader: batch 1/32 attempt 1: 18/20 returned [ok] -- retrying 2 missing in 20s
2026-09-25 13:48:18,134 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 13:48:18,140 ERROR data_engineering.eod_data.refinitiv: Refinitiv get_data failed for chunk starting at 0: Unable to resolve all requested identifiers in ['ABMD.OQ', 'AGN.N'].
2026-09-25 13:48:18,141 WARNING data_engineering.loaders.eod_loader: batch 1/32 attempt 2: 18/20 returned [ok] -- retrying 2 missing in 20s
2026-09-25 13:48:40,754 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026

EodLoadSummary(rows_written=974869, securities_requested=638, securities_loaded=606, dropped_implausible_rows=345, missing_security_ids=[6, 20, 22, 32, 55, 105, 149, 153, 164, 171, 172, 179, 210, 238, 243, 326, 343, 409, 410, 419, 449, 481, 501, 505, 511, 546, 563, 581, 606, 617, 624, 626])


### Gap recovery, holiday pruning and coverage

* `mop_up_missing_prices` re-fetches only securities with fewer than 50 priced days in the window (names that joined
  late are left alone). Whatever is still missing afterwards is typically delisted / renamed and is reported.
* Holdings were generated for every weekday, but exchanges close on holidays; `prune_low_coverage_holdings`
  removes SP500/SPY/GSPC holdings on dates where fewer than half the names have a price, so the index is never
  renormalised onto two or three stocks.

In [8]:
mop_up = mop_up_missing_prices(universe, session, engine, START_DATE, END_DATE, vendor="refinitiv")
print("mop-up:", mop_up)

pruned = prune_low_coverage_holdings(engine, ["SP500", "SPY", "GSPC"], START_DATE, END_DATE, min_coverage=0.5)
print(f"pruned {len(pruned)} low-coverage dates:", pruned[:10], "..." if len(pruned) > 10 else "")

coverage = coverage_report(engine, "SP500")
print(f"SP500 price coverage: mean {coverage['coverage_pct'].mean():.1f}% | min {coverage['coverage_pct'].min():.1f}% "
      f"on {pd.to_datetime(coverage.loc[coverage['coverage_pct'].idxmin(), 'as_of_date']).date()} | "
      f"{int((coverage['priced'] == coverage['holdings']).sum())}/{len(coverage)} dates at 100%")

2026-09-25 14:43:47,601 INFO data_engineering.loaders.eod_loader: mop-up: 33/638 securities broken/empty in 2020-01-01..2026-08-31
2026-09-25 14:43:47,602 INFO data_engineering.loaders.eod_loader: EOD pull: 33 securities in 2 batches of 20
2026-09-25 14:43:56,496 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 14:44:14,029 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 14:44:14,406 INFO data_engineering.eod_data.refinitiv: symbols with no data: ['ABMD.OQ', 'AGN.N', 'AIRC.N', 'ALXN.OQ', 'ATVI.OQ', 'CERN.OQ', 'CTXS.OQ', 'CXO.N', 'DFS.N', 'DISCK.OQ', 'DISH.OQ', 'DRE.N', 'ETFC.OQ', 'FLIR.OQ', 'FRC.N', 'JWN.N', 'KSU.N', 'MXIM.OQ', 'NBL.OQ']
2026-09-25 14:44:14,410 WARNING data_engineering.loaders.eod_loader: batch 1/2 attempt 1: 1/20 returned [ok] -- retrying 19 missing in 20s
2026-09-25 14:44:37,763 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 14:44:37,769 ERROR

mop-up: EodLoadSummary(rows_written=45, securities_requested=33, securities_loaded=1, dropped_implausible_rows=0, missing_security_ids=[6, 20, 22, 32, 55, 105, 149, 153, 164, 171, 172, 179, 210, 238, 243, 326, 343, 409, 410, 419, 449, 481, 501, 505, 511, 546, 563, 581, 606, 617, 624, 626])


2026-09-25 14:50:06,616 INFO data_engineering.loaders.eod_loader: pruned holdings on 66 low-coverage dates


pruned 66 low-coverage dates: ['2020-01-01', '2020-01-02', '2020-01-20', '2020-02-17', '2020-04-10', '2020-05-25', '2020-07-03', '2020-09-07', '2020-11-26', '2020-12-25'] ...
SP500 price coverage: mean 98.0% | min 89.3% on 2020-03-24 | 275/1673 dates at 100%


### Optional: repair known-fragile prices in a fresh process

After a long notebook session the in-kernel LSEG pull has occasionally returned corrupted prices for a few
instruments (a share-class scale error, an index level offset). `toolkit/scripts/repair_eod_prices.py` re-pulls
named securities in a fresh interpreter and upserts them.

In [9]:
import subprocess
import sys

REPAIR = False   # set True if the checks in notebook 02 show a suspicious series
if REPAIR:
    proc = subprocess.run([sys.executable, "../scripts/repair_eod_prices.py", START_DATE, END_DATE], capture_output=True, text=True, timeout=900)
    print(proc.stdout[-3000:], proc.stderr[-1000:])

## 1.7 Verify

In [10]:
shares = database.read_security_fundamentals(session, engine, "shares_outstanding")
print(f"shares_outstanding rows: {len(shares):,} for {shares['security_id'].nunique()} securities")
market_data = database.read_market_data(session, engine, START_DATE, END_DATE)
print(f"market_data rows: {len(market_data):,} for {market_data['security_id'].nunique()} securities, "
      f"{market_data['as_of_date'].nunique()} dates")
market_data.head()

shares_outstanding rows: 998,660 for 636 securities
market_data rows: 974,869 for 606 securities, 1674 dates


,md_id,as_of_date,security_id,open,high,low,close,adj_close,volume,dividends,stock_splits,interval,dataload_date,price_currency,source_vendor
0,1,2020-01-03,1,84.67,85.32,84.51,84.57,84.57,378070,0.0,0.0,1d,2026-09-25 13:47:55,USD,Refinitiv
1,2,2020-01-06,1,84.00,84.82,83.78,84.82,84.82,683281,0.0,0.0,1d,2026-09-25 13:47:55,USD,Refinitiv
2,3,2020-01-07,1,84.29,85.26,83.96,85.08,85.08,390892,0.0,0.0,1d,2026-09-25 13:47:55,USD,Refinitiv
3,4,2020-01-08,1,85.96,86.45,85.28,85.92,85.92,596307,0.0,0.0,1d,2026-09-25 13:47:55,USD,Refinitiv
4,5,2020-01-09,1,86.46,87.70,86.17,87.27,87.27,528346,0.0,0.0,1d,2026-09-25 13:47:55,USD,Refinitiv


In [11]:
session.close()
connection.close()

**Next:** open `02_backtest_sp500_reconstruction.ipynb` to compare the reconstructed index with SPY / ^GSPC and
backtest the MAG8 book.